# Plain Instructions to Code: IIR Low-Pass Filter on IQ Signals

This notebook directly implements the specifications outlined in `prompts/iir-plain-instructions-to-code.md`.

---

### Requirements Checklist
- [x] **1. Libraries**: Import `numpy`, `scipy.signal`, `matplotlib.pyplot`.
- [x] **2. Reproducibility**: Set `SEED = 42`.
- [x] **3. Data Generation**: Synthetic IQ signal ($N = 5, L = 1000$) with canonical layout `(N, 2, L)`.
- [x] **4. Filter Design**: 2nd-order Butterworth IIR low-pass filter with normalized cutoff $0.1$ via `scipy.signal.butter`.
- [x] **5. Filtering**: Apply filter using `scipy.signal.lfilter` along the time axis (`axis=2`).
- [x] **6. Power Calculation**: Compute average power before and after ($P = \text{mean}(I^2 + Q^2)$).
- [x] **7. Visualizations**: 4 subplots: (a) I time trace, (b) Q time trace, (c) I/Q constellation, (d) power comparison.
- [x] **8. Persistence**: Save filtered output to `'filtered_iq.npz'`.
- [x] **9. Verification**: Print shape, dtype, power before, power after, and power ratio.

## Steps 1 & 2: Imports and Reproducibility Setup

In [1]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)
print("Environment initialized with SEED = 42.")

Environment initialized with SEED = 42.


## Step 3: Create Synthetic IQ Data with Canonical Layout `(N, 2, L)`

- $N = 5$ examples
- $L = 1000$ time samples per example
- Axis 0: examples, Axis 1: I/Q components (0 = I, 1 = Q), Axis 2: time samples

In [2]:
N = 5
L = 1000
t = np.linspace(0, 1, L, endpoint=False, dtype=np.float32)

# Low frequency tone (f0 = 5 Hz)
f0 = 5.0
tone_i = np.cos(2 * np.pi * f0 * t, dtype=np.float32)
tone_q = np.sin(2 * np.pi * f0 * t, dtype=np.float32)

# Broadcast across N batches and inject high-frequency Gaussian noise
I = np.repeat(tone_i[np.newaxis, :], N, axis=0) + 0.5 * rng.standard_normal((N, L)).astype(np.float32)
Q = np.repeat(tone_q[np.newaxis, :], N, axis=0) + 0.5 * rng.standard_normal((N, L)).astype(np.float32)

# Form canonical tensor layout
X = np.stack([I, Q], axis=1)  # shape: (5, 2, 1000)

print(f"X.shape: {X.shape}")
print(f"X.dtype: {X.dtype}")

X.shape: (5, 2, 1000)
X.dtype: float32


## Steps 4 & 5: Design and Apply 2nd-Order Butterworth IIR Filter

- Order $M = 2$
- Normalized cutoff frequency $W_n = 0.1$ (Nyquist frequency normalized to $1.0$)
- Applied via `scipy.signal.lfilter` along time axis (`axis=2`)

In [3]:
# Design 2nd-order Butterworth low-pass filter
order = 2
cutoff = 0.1
b, a = signal.butter(N=order, Wn=cutoff, btype='lowpass')

# Apply filter along time axis (axis=2) separately for I and Q components
X_filtered = signal.lfilter(b, a, X, axis=2).astype(np.float32)

print(f"Numerator coefficients (b):   {b}")
print(f"Denominator coefficients (a): {a}")
print(f"X_filtered shape:            {X_filtered.shape}")

Numerator coefficients (b):   [0.02008337 0.04016673 0.02008337]
Denominator coefficients (a): [ 1.         -1.56101808  0.64135154]
X_filtered shape:            (5, 2, 1000)


## Step 6: Compute Average Power

$$P = \text{mean}(I^2 + Q^2)$$

In [4]:
power_before = float(np.mean(X[:, 0, :]**2 + X[:, 1, :]**2))
power_after = float(np.mean(X_filtered[:, 0, :]**2 + X_filtered[:, 1, :]**2))
power_ratio = power_after / power_before

print(f"Power before filtering: {power_before:.5f}")
print(f"Power after filtering:  {power_after:.5f}")
print(f"Power ratio (P_after / P_before): {power_ratio:.5f}")

Power before filtering: 1.51570
Power after filtering:  1.05798
Power ratio (P_after / P_before): 0.69802


## Step 7: Visualizations (4 Subplots)

(a) In-phase (I) time trace
(b) Quadrature (Q) time trace
(c) I/Q constellation diagram
(d) Power comparison bar chart

In [5]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# (a) I time trace
axes[0, 0].plot(t[:150], X[0, 0, :150], label="Raw I", color="lightsteelblue", alpha=0.8)
axes[0, 0].plot(t[:150], X_filtered[0, 0, :150], label="Filtered I", color="navy", linewidth=2)
axes[0, 0].set_title("(a) In-phase (I) Time Trace (First 150 Samples)")
axes[0, 0].set_xlabel("Time [s]")
axes[0, 0].set_ylabel("Amplitude")
axes[0, 0].legend()
axes[0, 0].grid(True, linestyle="--", alpha=0.6)

# (b) Q time trace
axes[0, 1].plot(t[:150], X[0, 1, :150], label="Raw Q", color="moccasin", alpha=0.8)
axes[0, 1].plot(t[:150], X_filtered[0, 1, :150], label="Filtered Q", color="darkorange", linewidth=2)
axes[0, 1].set_title("(b) Quadrature (Q) Time Trace (First 150 Samples)")
axes[0, 1].set_xlabel("Time [s]")
axes[0, 1].set_ylabel("Amplitude")
axes[0, 1].legend()
axes[0, 1].grid(True, linestyle="--", alpha=0.6)

# (c) I/Q constellation
axes[1, 0].scatter(X[0, 0, :], X[0, 1, :], s=10, alpha=0.4, color="crimson", label="Raw IQ")
axes[1, 0].scatter(X_filtered[0, 0, :], X_filtered[0, 1, :], s=10, alpha=0.6, color="teal", label="Filtered IQ")
axes[1, 0].set_title("(c) I/Q Constellation Comparison")
axes[1, 0].set_xlabel("In-phase (I)")
axes[1, 0].set_ylabel("Quadrature (Q)")
axes[1, 0].legend()
axes[1, 0].grid(True, linestyle="--", alpha=0.6)
axes[1, 0].axis("equal")

# (d) Power comparison
labels = ["Before", "After"]
powers = [power_before, power_after]
colors = ["#d9534f", "#5cb85c"]
bars = axes[1, 1].bar(labels, powers, color=colors, width=0.5)
axes[1, 1].set_title("(d) Average Signal Power Comparison")
axes[1, 1].set_ylabel("Power (Mean Amplitude Squared)")
axes[1, 1].grid(True, linestyle="--", alpha=0.6, axis="y")
for bar in bars:
    yval = bar.get_height()
    axes[1, 1].text(bar.get_x() + bar.get_width()/2.0, yval + 0.02, f"{yval:.4f}", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()

## Steps 8 & 9: Save NPZ and Print Formal Verification

In [6]:
# Step 8: Save result to 'filtered_iq.npz'
np.savez("filtered_iq.npz", X_filtered=X_filtered, X_raw=X)
print("Filtered array saved to 'filtered_iq.npz'.")

# Step 9: Print formal verification
print("\n=== FORMAL VERIFICATION REPORT ===")
print(f"Shape:        {X_filtered.shape}")
print(f"Dtype:        {X_filtered.dtype}")
print(f"Power before: {power_before:.5f}")
print(f"Power after:  {power_after:.5f}")
print(f"Power ratio:  {power_ratio:.5f}")

# Assertions
assert X_filtered.shape == (5, 2, 1000), f"Shape mismatch: {X_filtered.shape}"
assert X_filtered.dtype == np.float32, f"Dtype mismatch: {X_filtered.dtype}"
assert power_after < power_before, "Power was not attenuated!"
print("\nAll verification checks PASSED.")

Filtered array saved to 'filtered_iq.npz'.
=== FORMAL VERIFICATION REPORT ===
Shape:        (5, 2, 1000)
Dtype:        float32
Power before: 1.51570
Power after:  1.05798
Power ratio:  0.69802
All verification checks PASSED.
